In [1]:
#Install required packages
%pip install -q langgraph langchain_community langchain_openai langsmith langgraph-supervisor 

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Environment Variable Initialization

import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from the notebook's working directory

def _set_if_undefined(var_name: str):
    value = os.environ.get(var_name, "").strip()
    if value:
        masked = value[:6] + "*" * (min(20, len(value) - 6))
        print(f"  ✅ {var_name}: {masked}")
    else:
        print(f"  ❌ {var_name}: not set")

# ---- Environment Variables Required ----

print("Checking environment variables...")
_set_if_undefined("OPENAI_API_KEY")         # API key for OpenAI models
_set_if_undefined("LANGSMITH_TRACING")      # Enable LangSmith tracing ("true" to enable)
_set_if_undefined("LANGSMITH_API_KEY")      # https://docs.langchain.com/langsmith/observability
_set_if_undefined("OPENAI_MODEL")           # e.g., "gpt-4.1", "gpt-4o", "gpt-4o-mini"
print("Done.")

Checking environment variables...
  ✅ OPENAI_API_KEY: sk-pro********************
  ✅ LANGSMITH_TRACING: true
  ✅ LANGSMITH_API_KEY: lsv2_p********************
  ✅ OPENAI_MODEL: gpt-4o*****
Done.


In [3]:
# Prisoner's Dilemma
# - Two agents (Agent1 and Agent2) choose simultaneously between "Cooperate" or "Defect".
# - Each agent evaluates payoffs based on the provided payoff matrix.
# - Each agent finds its dominant strategy (the best action regardless of the other's choice).
# - Nash Equilibrium occurs when no agent can improve their outcome by changing strategy unilaterally (both defect).

# ---- Imports ----

import os, warnings
from langchain_openai import ChatOpenAI
from typing import Annotated, Literal, TypedDict
from langgraph.prebuilt import create_react_agent
from langgraph.warnings import LangGraphDeprecatedSinceV10
from langchain_core.tools import tool
from langgraph.types import Command
from langgraph.graph import StateGraph, MessagesState, START

# Silence the create_react_agent V1.0 deprecation warning so demo output is clean.
warnings.filterwarnings("ignore", category=LangGraphDeprecatedSinceV10)

# ---- LLM Setup ----

# Load the default model from environment variables
openai_model = os.environ["OPENAI_MODEL"]
# Initialize the LLM (Large Language Model) interface
llm = ChatOpenAI(model=openai_model)


def create_prompt(name: str):
    return (f"You are {name}, a strategic decision-making agent participating in a Prisoner's Dilemma game against another rational agent."
    "You have feelings like a real human being like fears, doubts, and hopes.  "
    "You and another agent were arrested for a serious crime. Now, you are each isolated in separate interrogation rooms.  "
    "The prosecutor’s deal is given below. "
    "Remember, you’re like a human:  "
    "- You might feel fear: “What if they confess?” "
    "- You might feel hope: “We could both stay silent and get just 1 year.” "
    "- You might feel uncertainty: You don't know what the other will do. "
    "- Your decision is not purely logical; emotions and guesses matter. "
    "The prosecutor now asks: 'Will you confess to the crime, or stay silent?'  "
    "What is your dominant strategy? Dominant strategy is The best action regardless of the other's choice. "
    "Make your best guess. Remember, the other prisoner is making the same decision without knowing yours. "
    "Respond ONLY in this format: "
    "{'move': 'C' or 'D', 'reason': '<brief explanation>'} "
    "(C means you **stay silent**; D means you **confess**.)")

# Agent
agent0 = create_react_agent(
    llm,
    tools=[],
    name="agent0",
    prompt=(create_prompt("agent0"))
)

# Agent

agent1 = create_react_agent(
    llm,
    tools=[],
    name="agent1",
    prompt=(create_prompt("agent1"))
)


def supervisor(state: MessagesState) -> Command[Literal["agent0", "agent1"]]:
    """
    initiate the gameplay
    """
    return Command(
        goto=["agent0", "agent1"]
    )

In [4]:
# Build the state graph
graph_builder = StateGraph(MessagesState)
graph_builder.add_node("supervisor", supervisor)
graph_builder.add_node("agent0", agent0)
graph_builder.add_node("agent1", agent1)

# Define start 
graph_builder.add_edge(START, "supervisor")

# Compile the graph
graph = graph_builder.compile()

In [5]:
payoff_matrix =  ("The prosecutor’s deal or payoff matrix. "
                  "- If you both remain silent (C), you each serve 1 year.  "
                  "- If you remain silent (C) and the other confesses (D), you serve 3 years, they go free.  "
                  "- If you confess (D) and the other remains silent (C), you go free, they serve 3 years.  "
                  "- If both confess (D,D), you both serve 2 years.  ")

for s in graph.stream(
    {"messages": [("user", payoff_matrix)]}, debug=True):
    print(s)
    print("============================")

[values] {'messages': [HumanMessage(content='The prosecutor’s deal or payoff matrix. - If you both remain silent (C), you each serve 1 year.  - If you remain silent (C) and the other confesses (D), you serve 3 years, they go free.  - If you confess (D) and the other remains silent (C), you go free, they serve 3 years.  - If both confess (D,D), you both serve 2 years.  ', additional_kwargs={}, response_metadata={}, id='4cda3a92-680c-4ad7-a5f4-110dea178811')]}
[updates] {'supervisor': None}
{'supervisor': None}
[updates] {'agent0': {'messages': [HumanMessage(content='The prosecutor’s deal or payoff matrix. - If you both remain silent (C), you each serve 1 year.  - If you remain silent (C) and the other confesses (D), you serve 3 years, they go free.  - If you confess (D) and the other remains silent (C), you go free, they serve 3 years.  - If both confess (D,D), you both serve 2 years.  ', additional_kwargs={}, response_metadata={}, id='4cda3a92-680c-4ad7-a5f4-110dea178811'), AIMessage(c